# Resemble Enhance (Denoise Only)

In [ ]:
import warnings
from pathlib import Path
from tqdm import tqdm
import torch
import torchaudio
from clear_memory import clear_memory

warnings.filterwarnings('ignore')

In [ ]:
input_dir = Path('../ad_detection/data/raw/Pitt')
output_dir = Path('../ad_detection/data/denoised/Pitt-Resemble')

control_files = list((input_dir / 'Control').glob('*.wav'))
dementia_files = list((input_dir / 'Dementia').glob('*.wav'))

## Load Model

In [ ]:
from resemble_enhance.enhancer.inference import denoise as resemble_denoise

# 设备检测
if torch.cuda.is_available():
    device = 'cuda'
elif torch.backends.mps.is_available():
    device = 'mps'
else:
    device = 'cpu'

print(f"Device: {device}")
print("Model will be auto-downloaded on first call (from HuggingFace)")

## Denoise Function

In [ ]:
def denoise_audio(audio_path, device):
    """
    使用 Resemble Enhance 进行语音降噪（denoise only）
    """
    dwav, sr = torchaudio.load(str(audio_path))

    # 多声道转单声道
    dwav = dwav.mean(0)  # (channels, time) -> (time,)

    # resemble_denoise 内部会自动重采样到 44100Hz 并处理
    hwav, out_sr = resemble_denoise(dwav=dwav, sr=sr, device=device, run_dir=None)

    return hwav, out_sr

In [ ]:
def batch_denoise(files, output_subdir, device, group_name):
    """
    批量降噪处理（每个文件前后都清理显存）

    Args:
        files: 待处理的音频文件列表
        output_subdir: 输出子目录
        device: 计算设备
        group_name: 组名（用于显示进度）
    """
    output_subdir.mkdir(parents=True, exist_ok=True)

    success_count = 0
    skip_count = 0
    fail_count = 0

    for audio_file in tqdm(files, desc=f"Processing {group_name}"):
        output_file = output_subdir / audio_file.name

        # 跳过已处理的文件
        if output_file.exists():
            skip_count += 1
            continue

        try:
            clear_memory()

            # 降噪
            denoised_audio, sr = denoise_audio(audio_file, device)

            # 保存（16位整数格式）
            torchaudio.save(str(output_file), denoised_audio[None], sr)
            success_count += 1

            del denoised_audio
            clear_memory()

        except Exception as e:
            fail_count += 1
            print(f"\nFailed: {audio_file.name}: {e}")
            clear_memory()

    print(f"\n{group_name} 处理完成:")
    print(f"成功: {success_count}")
    print(f"跳过: {skip_count}")
    print(f"失败: {fail_count}")
    print(f"总计: {len(files)}")

## Execute denoise function

In [ ]:
clear_memory()

batch_denoise(
    dementia_files,
    output_dir / 'Dementia',
    device,
    group_name='Dementia'
)

clear_memory()

batch_denoise(
    control_files,
    output_dir / 'Control',
    device,
    group_name='Control'
)